# 05 — Train the neural baselines

**What this notebook does.** Trains two from-scratch deep learning models — N-BEATS and a
DeepAR-class LSTM — on the same folds, the same horizons, and the same metrics as every
other model in the benchmark, then pushes their forecasts back to the Hub.

These are the honest comparison point for a foundation model. Chronos-2 arrives already
pretrained on millions of time series; these two start from nothing and see only our data.
The gap between them, and how that gap changes with training-set size, is the most
informative result in the project.

## Before you run this, these must already exist

| Thing | Where | Made by |
|---|---|---|
| Processed series | `rohanjain2312/forecastbench-data` → `processed/*.parquet` | build Step 15 |
| Colab Secrets | the 🔑 panel on the left | you, once |

Notebook 04 does **not** need to have finished first — these two are independent.

## The one rule this notebook follows

**No modelling logic lives here.** The fold loop, the early stopping, and the quantile
construction are all in `forecast_bench/`. Every cell imports and calls.

## Step 1 — Confirm the GPU

Runtime → Change runtime type → **H100**.

In [ ]:
!nvidia-smi

## Step 2 — Install the project

Installed from a source tarball rather than `git+https://...`: pip's git codepath shells out to the container's git binary, which has been flaky inside Colab; fetching the tarball over plain HTTP avoids that dependency entirely.

`--force-reinstall --no-deps` matters as much as the tarball switch itself: without it, pip sees `forecast-bench` already installed (the version number never changes) and silently skips reinstalling, so reopening this notebook and clicking Run All can keep running *stale* code from an earlier session if Colab reconnects you to the same live runtime rather than a fresh one. `--no-deps` keeps this fast by leaving torch, darts and chronos-forecasting alone.

The install line below busts every caching layer between this notebook and GitHub — pip's own HTTP cache, and any proxy in between. A tarball URL alone was not enough in practice: this notebook was reopened fresh from GitHub twice in a row and still ran stale code both times, because the URL never changes even though its content does, and something between here and GitHub kept serving what it had already fetched. Appending a fresh, per-run query parameter forces a genuine refetch every time, no matter what. The cell immediately after verifies it worked, rather than trusting it silently.


In [ ]:
import time

_cache_bust = int(time.time())
_tarball_url = (
    "https://github.com/Rohanjain2312/forecast_bench/archive/refs/heads/main.tar.gz"
    f"?_cb={_cache_bust}"
)

# Two installs, and both are load-bearing.
#
# 1. WITH dependencies. On a fresh VM this is what brings in pydantic-settings, darts,
#    chronos-forecasting and the rest. A --no-deps-only install leaves the package
#    importable in name only and dies on the first import.
# 2. The package alone, forced. pip will not reinstall a package whose version it already
#    considers satisfied, so this is what guarantees the *code* is current on a runtime
#    that has seen an earlier version. --no-deps keeps this one to a couple of seconds.
#
# The ?_cb=<timestamp> on the URL makes every run a different URL, so no HTTP cache
# between here and GitHub can serve a stale copy to either step.
!pip install -q "{_tarball_url}"
!pip install -q --force-reinstall --no-deps "{_tarball_url}"

## Step 2b — Verify the install actually updated

Checks for three recent fixes rather than trusting the install cell above. If any of these
raise, the install did not take effect — **Runtime → Restart session**, then run every cell
again from the top. Do not skip past this cell if it fails; everything below depends on
these being present.


In [ ]:
import inspect

# 2. The sample-efficiency window resolver, which Step 6 depends on.
from forecast_bench.config import enable_tensor_cores

# 1. regimes must tolerate a missing regimes.yaml. Only forecast_bench/ ships in the
#    package, so experiments/configs/ does not exist here at all. Stale code raises
#    FrozenThresholdError on this import.
from forecast_bench.evaluation.regimes import CALM_UPPER
from forecast_bench.models.base import sample_efficiency_window_size

assert CALM_UPPER == 15.9, f"unexpected frozen threshold: {CALM_UPPER}"
assert sample_efficiency_window_size("1y") > 512, "stale sample-efficiency sizing"
assert "precision" in inspect.signature(enable_tensor_cores).parameters

print("OK: forecast_bench install is current.")

## Step 3 — Load your credentials from Colab Secrets

Nothing is printed.

In [ ]:
import os

from google.colab import userdata

from forecast_bench.config import get_config

for key in ["HF_TOKEN", "FRED_API_KEY", "WANDB_API_KEY"]:
    try:
        os.environ[key] = userdata.get(key)
        print(f"{key}: loaded")
    except Exception:
        print(f"{key}: not set")

os.environ.setdefault("HF_DATASET_REPO", "rohanjain2312/forecastbench-data")
os.environ.setdefault("HF_MODEL_REPO", "rohanjain2312/forecastbench-chronos")

# get_config() is a process-wide singleton (functools.lru_cache) so that every
# caller in one run observes identical settings. That means if anything called
# it even once before this cell ran -- an earlier attempt, a re-run out of order --
# it is now permanently cached WITHOUT these secrets, and setting os.environ above
# would silently have no effect for the rest of this session. Clearing it here
# makes the notebook correct regardless of what was run before this cell.
get_config.cache_clear()

## Step 3b — Turn on Tensor Cores

The A100 can run matmuls on its Tensor Cores at roughly 2–5x the speed of full fp32, at
the cost of a 10-bit mantissa instead of 23. That precision loss is far below the noise
floor of a financial forecasting task, and it is the standard recommendation for neural
training.

This is off by default in this project and switched on deliberately here, because it
changes numerics and therefore has to apply **uniformly to everything being compared**.
Setting it once, before any training, means every model in this notebook — and every point
on the sample-efficiency curve — is trained under identical numerics.

Note that notebook 04's Chronos fine-tuning was run *without* this. That is fine: the two
notebooks produce separate model families that are compared through their forecasts, not
through their weights. What would not be fine is enabling it halfway through this notebook.


In [ ]:
from forecast_bench.config import enable_tensor_cores

precision = enable_tensor_cores("high")
print(f"matmul precision: {precision}")

## Step 4 — Run the backtest with the neural models included

For a neural model, "training" and "backtesting" are the same operation: the harness
retrains the network at each block boundary and forecasts forward from every fold origin.
So this single call trains the models *and* produces their forecasts, driven by exactly the
same runner that drives ARIMA and Chronos-2.

That matters more than it sounds. The models traverse identical code, so a difference in the
results table is a difference between models rather than a difference between two
evaluation scripts that were each written on a different afternoon.

In [ ]:
from forecast_bench.backtest.runner import run_series_backtest
from forecast_bench.backtest.writer import write_results
from forecast_bench.config import get_config, setup_logging
from forecast_bench.data.hub import push_forecasts

setup_logging("INFO")
config = get_config()
config.ensure_dirs()

spy = run_series_backtest(
    series="spy_logrv",
    cadence="matched",
    arm="A",
    include_neural=True,
    device="gpu",
)
print(f"{len(spy):,} forecast rows, models: {sorted(spy.model_id.unique())}")

# Save and publish immediately. Colab's disk is ephemeral -- a recycled runtime takes
# these hours with it -- and Step 7 is too late to be the first time this leaves the VM.
write_results(spy, config.forecasts_dir)
push_forecasts()
print("spy_logrv saved locally and pushed to the Hub")

## Step 5 — The same for the Treasury yield

The contrast series. Saved and pushed the moment it finishes, for the same reason as
Step 4: nothing that took an hour to compute should exist only on a disk that can vanish.


In [ ]:
dgs10 = run_series_backtest(
    series="dgs10",
    cadence="matched",
    arm="A",
    include_neural=True,
    device="gpu",
)
write_results(dgs10, config.forecasts_dir)
push_forecasts()
print(f"{len(dgs10):,} forecast rows for dgs10, saved and pushed")

## Step 6 — The sample-efficiency sweep

The same two models trained on 1 year, 3 years, and 10 years of data. Plotted against the
Chronos-2 curve from notebook 04, this is the test of whether pretraining actually buys
data efficiency or just buys a head start.

`"1y"` and friends do not mean literally one year of raw observations — with context length
fixed at 512 across every model in the study, a slice that short could not supply even one
training example. They mean that many distinct forecast origins' worth of *additional*
material; `sample_efficiency_window_size` does the conversion.

**The `full` point is reused from Step 4 rather than recomputed.** `full` means "no
truncation", so running it here would be a byte-identical repeat of the Step 4 call — hours
of GPU time to reproduce a number already in memory. Reusing it also guarantees the curve's
endpoint is exactly the headline result rather than a second run that might drift from it.


In [ ]:
import pandas as pd

from forecast_bench.models.base import sample_efficiency_window_size

# Step 4 already computed the untruncated run. Fall back to its saved parquet if this
# cell is run in a fresh kernel where `spy` is no longer in memory.
try:
    full_result = spy.copy()
except NameError:
    full_result = pd.read_parquet(
        config.forecasts_dir / "spy_logrv_armA_block_ys.parquet"
    )
full_result["training_window"] = "full"

sweep = [full_result]
for name in ("1y", "3y", "10y"):
    result = run_series_backtest(
        series="spy_logrv",
        cadence="matched",
        arm="A",
        include_neural=True,
        device="gpu",
        training_window_days=sample_efficiency_window_size(name),
    )
    result["training_window"] = name
    sweep.append(result)
    print(f"{name}: {len(result):,} rows")

sweep_frame = pd.concat(sweep, ignore_index=True)
# Written to forecasts/sample_efficiency/, not forecasts/ itself. The sweep's
# `full` slice repeats the Step 4 run exactly, and build_results globs forecasts/
# non-recursively -- keeping the sweep one level down is what stops those duplicate
# rows being counted into the headline table.
sweep_frame.to_parquet(
    config.sample_efficiency_dir / "spy_logrv.parquet", index=False
)
push_forecasts()
print(f"sweep total: {len(sweep_frame):,} rows across {sweep_frame.training_window.nunique()} windows, pushed")

## Step 7 — Push the forecasts back to the Hub

The GPU work happens here, but the scoring happens locally, through the same
`evaluation/aggregate.py` that scores every other model. Sending the forecasts back rather
than the metrics is deliberate: one scoring implementation, one set of numbers.

In [ ]:
from forecast_bench.data.hub import push_forecasts

for path in push_forecasts():
    print("uploaded", path)

## Done

**Tell Claude Code "done"** and it will pull these forecasts, run the full benchmark, and
apply the pre-registered losing condition to the result.